<a href="https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/Khadija-Azam05/ML-Projects.git

Cloning into 'ML-Projects'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 164 (delta 64), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.87 MiB | 13.60 MiB/s, done.
Resolving deltas: 100% (64/64), done.


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# Signal 1: days since the page was last updated

print("Days since last update:")
print(df["days_since_last_update"].describe())

print("\nMissing values:")
print(df["days_since_last_update"].isna().sum())

Days since last update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Missing values:
0


In [ ]:
stale_check = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, float("inf")],
    labels=["0-90", "91-180", "181-365", "365+"]
)

stale_table = (
    stale_check
    .value_counts(sort=False)
    .rename_axis("days_since_last_update")
    .reset_index(name="n")
)

print(stale_table)

  days_since_last_update      n
0                   0-90  20655
1                 91-180   9171
2                181-365    169
3                   365+      5


In [ ]:
print([col for col in df.columns if "refresh" in col.lower()])

[]


In [ ]:
print([col for col in df.columns if "flag" in col.lower()])

[]


In [ ]:
stale_summary = (
    df.assign(
        stale_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[-1, 90, 180, 365, float("inf")],
            labels=["0-90", "91-180", "181-365", "365+"]
        )
    )
    .groupby("stale_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        avg_impressions=("impressions_90d", "mean"),
        avg_clicks=("clicks_90d", "mean")
    )
    .reset_index()
)

print(stale_summary)

  stale_bucket      n  avg_impressions  avg_clicks
0         0-90  20655      4219.161317   13.693149
1       91-180   9171      7486.665140   21.766765
2      181-365    169      1206.893491    2.745562
3         365+      5         8.200000    0.200000


In [ ]:
volume_check = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 1000, 5000, float("inf")],
    labels=["0-100", "101-500", "501-1000", "1001-5000", "5000+"]
)

volume_summary = (
    df.assign(volume_bucket=volume_check)
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        avg_clicks=("clicks_90d", "mean")
    )
    .reset_index()
)

print(volume_summary)

  volume_bucket     n  avg_clicks
0         0-100  8006    0.146765
1       101-500  5279    0.586475
2      501-1000  3206    1.421397
3     1001-5000  7359    6.306563
4         5000+  6150   69.541789


### Signal 1: Days since last update — MIXED

I checked page activity across different update-age groups. The 91–180 day group had the highest average impressions and clicks, while the older groups had much lower activity. The oldest groups were also very small, so I don't want to treat age alone as a strong signal.

### Signal 2: Impressions over 90 days — CONFIRMED

The relationship was much clearer here. Average clicks increased as impressions increased, from 0.15 clicks in the 0–100 bucket to 69.54 clicks in the 5000+ bucket.

### Rule

I will rank pages mainly by search visibility. Older pages can receive a separate reason code when they also have strong visibility, but age will not be the main part of the score.

### Reason codes

- **high_visibility** — the page has high 90-day impressions.
- **stale_high_visibility** — the page has high impressions and has not been updated recently.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue

I am using 90-day impressions as the main score because the signal check showed a clear relationship with clicks. Pages that are also older will get the **stale_high_visibility** reason code.

In [ ]:
from pathlib import Path
import numpy as np

# Main score: higher visibility = higher priority
df["score"] = df["impressions_90d"]

# One reason code for each row
df["reason_code"] = np.where(
    (df["days_since_last_update"] > 90) & (df["impressions_90d"] >= 5000),
    "stale_high_visibility",
    "high_visibility"
)

# Action label
df["action"] = np.where(
    df["score"] > 0,
    "review",
    "no_action"
)

# Rank highest scores first
queue = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# Create the output folder if needed
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the ranked queue
queue.to_csv(output_path, index=False)

print("Top 20:")
print(
    queue[
        ["rank", "content_id", "score", "reason_code", "action"]
    ].head(20).to_string(index=False)
)

print("\nRows in queue:", len(queue))
print("Saved to:", output_path)

print("\nReason code counts:")
print(queue["reason_code"].value_counts())

print("\nAction counts:")
print(queue["action"].value_counts())

print("\nScore check:")
print(queue["score"].describe())

Top 20:
 rank           content_id  score           reason_code action
    1 content_5fe46e04994d 517715 stale_high_visibility review
    2 content_aaef01a50def 517109       high_visibility review
    3 content_8c19996aa890 509252       high_visibility review
    4 content_2cb567c3c89b 497727       high_visibility review
    5 content_4c36c775b818 463103       high_visibility review
    6 content_2dba2b1f9536 443434 stale_high_visibility review
    7 content_1a9e894be2e2 416180       high_visibility review
    8 content_2c2606c5d176 347399 stale_high_visibility review
    9 content_db5989a78dd3 345111       high_visibility review
   10 content_44e481c8f55b 312694       high_visibility review
   11 content_cb112fce36be 309910 stale_high_visibility review
   12 content_9532f197bbc8 309192 stale_high_visibility review
   13 content_36ff89c8214e 295097 stale_high_visibility review
   14 content_8e7ba84a972b 288426       high_visibility review
   15 content_b28d1efd668f 286608 stale_high_vi

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 rows using the score, reason code, impressions, clicks, update age, and average position.

| Rank | Action | Reason                | Confidence | What could make it wrong?                                                                             |
| ---: | ------ | --------------------- | ---------- | ----------------------------------------------------------------------------------------------------- |
|    1 | review | stale_high_visibility | High       | High impressions do not guarantee the page needs an update.                                           |
|    2 | review | high_visibility       | High       | The page was updated recently, so a refresh may not be needed.                                        |
|    3 | review | high_visibility       | High       | High visibility alone does not show that the content needs changing.                                  |
|    4 | review | high_visibility       | Medium     | Its average position is 22.2, so impressions alone may not be enough to prioritize it.                |
|    5 | review | high_visibility       | High       | The page is already ranking well, so a refresh may not add much value.                                |
|    6 | review | stale_high_visibility | Medium     | Its average position is 27.9, so high impressions may not mean it is a good refresh candidate.        |
|    7 | review | high_visibility       | High       | The page was updated recently.                                                                        |
|    8 | review | stale_high_visibility | High       | The page has strong visibility and is older, but age does not prove that an update is needed.         |
|    9 | review | high_visibility       | High       | The page was updated recently.                                                                        |
|   10 | review | high_visibility       | High       | The page already has a strong average position of 1.4.                                                |
|   11 | review | stale_high_visibility | Medium     | Its clicks are relatively low compared with its very high impressions.                                |
|   12 | review | stale_high_visibility | High       | The high click count suggests the page is already performing well, so a refresh may not be necessary. |
|   13 | review | stale_high_visibility | Medium     | Only 154 clicks were recorded despite high impressions.                                               |
|   14 | review | high_visibility       | High       | The page was updated recently, so the staleness part of the rule does not apply.                      |
|   15 | review | stale_high_visibility | Low        | Its average position is 26.2 and clicks are low, so the score may be oversimplifying the opportunity. |
|   16 | review | high_visibility       | High       | The page was updated recently and already has good visibility.                                        |
|   17 | review | high_visibility       | Low        | It has high impressions but only 75 clicks, so impressions alone may over-prioritize it.              |
|   18 | review | high_visibility       | High       | The page was updated recently, so it may not need a refresh.                                          |
|   19 | review | high_visibility       | Medium     | The page has good visibility but only 605 clicks.                                                     |
|   20 | review | stale_high_visibility | Low        | It has high impressions but only 129 clicks and an average position of 26.2.                          |


In [ ]:
top20 = queue.head(20).copy()

print(
    top20[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "impressions_90d",
            "days_since_last_update",
            "clicks_90d",
            "avg_position"
        ]
    ].to_string(index=False)
)

 rank           content_id  score           reason_code action  impressions_90d  days_since_last_update  clicks_90d  avg_position
    1 content_5fe46e04994d 517715 stale_high_visibility review           517715                     104         741           4.2
    2 content_aaef01a50def 517109       high_visibility review           517109                      22        1270           5.4
    3 content_8c19996aa890 509252       high_visibility review           509252                      20         785           2.5
    4 content_2cb567c3c89b 497727       high_visibility review           497727                      48         487          22.2
    5 content_4c36c775b818 463103       high_visibility review           463103                      20        1889           2.3
    6 content_2dba2b1f9536 443434 stale_high_visibility review           443434                     104         910          27.9
    7 content_1a9e894be2e2 416180       high_visibility review           416180           

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

The main weakness I found is that impressions alone can produce some weak picks.

Rank 17 had 272,144 impressions but only 75 clicks, and rank 20 had 233,561 impressions but only 129 clicks. Rank 15 was also a weak-looking pick because it had 286,608 impressions but only 169 clicks and an average position of 26.2.

This means high visibility does not always mean that a page should be reviewed first. Position and click performance could improve the rule in a later version.

I did not use **trend_pct**, **trend_direction**, or **is_declining_label** in the score. I also did not use future-window information.

In [ ]:
# Check which columns the baseline actually uses
rule_features = [
    "impressions_90d",
    "days_since_last_update"
]

forbidden_features = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

print("Features used by the rule:")
print(rule_features)

print("\nForbidden features used:")
print([col for col in rule_features if col in forbidden_features])

print("\nWeak picks from the top 20:")
print(
    queue.loc[
        queue["rank"].isin([15, 17, 20]),
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "impressions_90d",
            "clicks_90d",
            "avg_position"
        ]
    ].to_string(index=False)
)

Features used by the rule:
['impressions_90d', 'days_since_last_update']

Forbidden features used:
[]

Weak picks from the top 20:
 rank           content_id  score           reason_code  impressions_90d  clicks_90d  avg_position
   15 content_b28d1efd668f 286608 stale_high_visibility           286608         169          26.2
   17 content_8451fc6f034d 272144       high_visibility           272144          75           2.3
   20 content_813e88069237 233561 stale_high_visibility           233561         129          26.2
